In [ ]:
from dataclasses import dataclass

import pandas as pd
import requests

from kibad_crawler.config import RAW_DATA_DIR
from kibad_crawler.semantic_scholar import datasets
from kibad_crawler.papers import PaperSearch, get_papers

In [ ]:
release_id = datasets.get_latest_release()
print(release_id)

# Publication Venues

In [ ]:
import time

links_publication_venues = datasets.download_links(release_id, "publication-venues")
time.sleep(2)
output_path = datasets.download_dataset_from_semantic_scholar(
    url=links_publication_venues["files"][0],
    output_path=RAW_DATA_DIR / "semantic_scholar" / "release" / release_id / "publication_venues",
)
df_venues = datasets.read_publication_venues(file_path=output_path)

In [ ]:
df_venues.info()

In [ ]:
from pathlib import Path

path_journals = Path("/Users/darenasc/projects/dfki/kibad-crawler/data/raw/journals.csv")


def get_issn_codes(path_journals_csv: Path) -> list:
    """Return a list of ISSN code to filter journals. It reads a CSV with ISSN
    column names and filter it's content.

    Args:
        file_path (Path): CSV file

    Returns:
        list: _description_
    """
    df_journals = pd.read_csv(path_journals_csv)
    columns = [x for x in df_journals.columns if "issn" in str(x).lower()]
    issn = []
    for column in columns:
        issn += [x for x in df_journals[column].unique() if isinstance(x, str)]
    return issn


issn = get_issn_codes(path_journals)
publication_venue_ids = df_venues[df_venues["issn"].isin(issn)].id.unique()
print(issn[:5])
print(len(publication_venue_ids))

In [ ]:
search_params = PaperSearch(publication_venue_ids=publication_venue_ids)
len(search_params.publication_venue_ids)

# Papers

In [ ]:
links_papers = datasets.download_links(release_id=release_id, dataset_name="papers")

In [ ]:
output_path_papers = RAW_DATA_DIR / "semantic_scholar" / "release" / release_id / "papers"

In [ ]:
for url in links_papers["files"][:4]:
    get_papers(url=url, output_path=output_path_papers, search_params=search_params)